![Python Banner](../../Image/05_06_Baysian_CausalML.png)



# 6.2 Graphs and Bayesian Networks



A causal effect is identified only after the graph says which variables may enter the adjustment set, the mediator set, or the instrument list. This notebook is the detail chapter for that family in `pycausalml.bayesian_causalML`:

| Class | Role |
|---|---|
| `CausalGraph` | DAG, d-separation, backdoor sets, frontdoor sets, instruments |
| `GaussianBayesianNetwork` | Linear-Gaussian structural equations and $E[Y \mid do(T)]$ |
| `DiscreteBayesianNetwork` | Dirichlet–multinomial conditionals, `do` versus evidence |
| `NaiveAdjustmentRegressor` | Bayesian outcome regression on a backdoor set read from the graph |
| `NaiveIVRegressor` | Two-stage least squares with instruments read from the graph |

The estimators are the dependency-free counterparts of the adjustment and naive-IV operations in [pgmpy](https://pgmpy.org/api/causal_inference.html) and of the fit-and-intervene workflow associated with CausalNex. CausalNex was discontinued on 24 June 2026; this module does not import it.



## Overview

### Why start with a graph?

Regression of $Y$ on $T$ estimates $E[Y \mid T]$. An intervention asks for $E[Y \mid do(T)]$. Those two agree when every open path from $T$ to $Y$ is a directed causal path. A common cause, a conditioned collider, or a mediator left out of the graph changes the functional.

The graph in this module is a directed acyclic graph. An edge `(parent, child)` means the parent is a direct cause of the child. Once the graph is drawn, identification treats it as causally sufficient: a latent confounder has to appear as a node, or the adjustment set is the adjustment set of a different system. A cycle raises `ValueError`.

### What this tutorial covers

- d-separation on the chain, the fork, and the collider
- The backdoor criterion, the frontdoor criterion, and the definition of an instrument
- A linear-Gaussian network whose total effect is known, compared with the naive regression
- A discrete network in which `do(T)` and conditioning on $T$ answer different questions
- The frontdoor functional, computed from $(T, M, Y)$ alone and checked against a `do` query
- Backdoor regression and graph-based instrumental variables on simulated systems with a known effect
- The Sachs flow-cytometry measurements in `Data/sachs_data.csv`, with a published protein-signaling skeleton



## The algorithms

### 1. d-separation

Let $G$ be a DAG. A path between $X$ and $Y$ is **blocked** by a conditioning set $Z$ when one of these holds:

- the path contains a chain $A \rightarrow B \rightarrow C$ or a fork $A \leftarrow B \rightarrow C$ and $B \in Z$
- the path contains a collider $A \rightarrow B \leftarrow C$ and neither $B$ nor any descendant of $B$ is in $Z$

$X$ and $Y$ are d-separated by $Z$ when every path between them is blocked. `CausalGraph.d_separated` implements the equivalent moral-graph test: take the ancestors of $X \cup Y \cup Z$, moralize that subgraph (join co-parents, drop arrowheads), delete $Z$, and ask whether any node of $X$ still meets any node of $Y$.

The three canonical graphs fix the reading of every later criterion.

| Graph | Edges | $A \perp C$ with $Z = \emptyset$ | $A \perp C$ given $B$ |
|---|---|---|---|
| Chain | $A \rightarrow B \rightarrow C$ | open | blocked |
| Fork | $A \leftarrow B \rightarrow C$ | open | blocked |
| Collider | $A \rightarrow B \leftarrow C$ | blocked | open |

Conditioning on a collider, or on a descendant of a collider, opens a path that was closed.

### 2. Backdoor adjustment

A set $Z$ satisfies the **backdoor criterion** for $(T, Y)$ when

1. $Z$ contains neither $T$ nor $Y$
2. no node of $Z$ is a descendant of $T$
3. $Z$ blocks every path from $T$ to $Y$ in the graph with the arrows out of $T$ removed

Condition 3 is d-separation in the mutilated graph. Removing the arrows out of $T$ deletes the causal paths, so the test only looks at backdoor paths.

When $Z$ qualifies and the conditional expectations exist,

$$
E[Y \mid do(T=t)] = \sum_z E[Y \mid T=t, Z=z] \, P(Z=z).
$$

`adjustment_sets` searches the empty set, the parents of $T$, and other subsets of the non-descendants of $T$. It returns the inclusion-minimal sets that pass the criterion. `minimal_adjustment_set` returns one smallest set. An empty result means the observed graph does not identify the effect by adjustment.

### 3. Frontdoor criterion

A mediator set $M$ satisfies Pearl's **frontdoor criterion** for $(T, Y)$ when

1. $M$ intercepts every directed path from $T$ to $Y$
2. there is no unblocked backdoor path from $T$ to $M$
3. $T$ blocks every backdoor path from $M$ to $Y$

The identifying functional uses only the observational distribution of $(T, M, Y)$:

$$
P(y \mid do(t)) = \sum_m P(m \mid t) \sum_{t'} P(y \mid m, t') \, P(t').
$$

`frontdoor_sets` returns mediator sets that meet the criterion. The class identifies the set. The notebook computes the functional from the sample, because a latent common cause of $T$ and $Y$ is not entered into the regression.

### 4. Instruments read from the graph

A node $Z$ is an **instrument** for $(T, Y)$ when there is a directed path from $Z$ to $T$, and $Z$ is d-separated from $Y$ in the graph with $T$ removed. The second clause is the exclusion restriction as drawn: every directed path from $Z$ to $Y$ goes through $T$. `instruments` returns those nodes. A direct edge $Z \rightarrow Y$ removes $Z$ from the list.

### 5. Linear-Gaussian Bayesian network

Each node has the structural assignment

$$
X_i = a_i + \sum_{j \in \mathrm{pa}(i)} \beta_{ij} X_j + \varepsilon_i, \qquad \varepsilon_i \sim N(0, \sigma_i^2),
$$

with independent noise across equations. `GaussianBayesianNetwork.fit` places a weak ridge prior on the slopes (the intercept is almost unpenalized) and returns the conjugate normal posterior of each equation. The noise variance is the residual variance, so the coefficient posterior is the usual conditional normal posterior given that variance.

A hard intervention $do(T=t)$ replaces the equation for $T$ with the constant $t$. In a linear system the average treatment effect is the total derivative along directed paths:

$$
E[Y \mid do(T=t_1)] - E[Y \mid do(T=t_0)] = (t_1 - t_0) \sum_{\text{paths } T \rightsquigarrow Y} \prod_{(j \rightarrow i) \in \text{path}} \beta_{ij}.
$$

`ate` evaluates that derivative by a topological accumulation, once at the posterior means and once for each draw of the coefficient vector. `sample(do=...)` draws from the posterior-mean equations with the intervened node clamped. The draws are a predictive check of the same linear effect. They are not a second posterior.

### 6. Discrete Bayesian network

Each conditional distribution is categorical. With a Dirichlet prior of concentration $\alpha$ on every parent configuration, the posterior mean of a cell is

$$
\hat P(X_i = k \mid \mathrm{pa}(i) = j) = \frac{n_{jk} + \alpha}{n_{j\cdot} + \alpha \, |\mathcal{X}_i|}.
$$

`query` marginalizes this conditional probability table. Exact enumeration is used when the joint table has at most `exact_limit` cells (default 8192). Larger joints use likelihood weighting. A `do` map replaces that node's conditional with a point mass and does not multiply the query weight by the observational conditional. An `evidence` map keeps the conditional and reweights by it. A node cannot be both.

For a numeric outcome,

$$
\mathrm{ATE} = E[Y \mid do(T=t_1)] - E[Y \mid do(T=t_0)].
$$

`ate` returns that contrast. It does not return a posterior interval: the conditional probabilities are posterior means, and the query is a function of those means.

### 7. Naive adjustment and naive instrumental variables

`NaiveAdjustmentRegressor` calls `minimal_adjustment_set` and fits

$$
Y = a + \tau T + Z^\top \gamma + \varepsilon
$$

with the same ridge-normal posterior. The reported effect is the posterior of $\tau$. Under the graph, linearity, and constant effects, $\tau$ is $E[Y \mid do(T=1)] - E[Y \mid do(T=0)]$.

`NaiveIVRegressor` calls `instruments`. If the list is empty it raises `ValueError`. Otherwise it fits Bayesian two-stage least squares: Dirichlet weights on the rows, weighted least squares of $T$ on $Z$, then weighted least squares of $Y$ on the fitted $\hat T$. The first-stage uncertainty is inside the second-stage draws. This is the native counterpart of pgmpy's `NaiveIVRegressor`. Covariates that are not instruments are not entered; a confounder that the graph says must be controlled belongs in `NaiveAdjustmentRegressor` or in `BayesianStructuralSCM`.



## Setup

This notebook fits the graph and network classes from **[PyCausalML](https://github.com/zia207/PyCausalML)** (`pycausalml.bayesian_causalML`). The setup finds the repository, imports the package, and installs the editable checkout only if the import fails.


In [ ]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "pycausalml").is_dir():
            return cand
    raise FileNotFoundError(
        "PyCausalML repository was not found. Run this notebook from the repository."
    )

root = _repo_root()
data_dir = root / "Data"
if not data_dir.is_dir():
    raise FileNotFoundError(
        "The Data directory was not found. It should sit next to pyproject.toml."
    )

try:
    import pycausalml  # noqa: F401
    print("PyCausalML ready.")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[plot]"],
    )
    print("PyCausalML installed.")
print("Data directory:", data_dir)


In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pycausalml.bayesian_causalML import (
    CausalGraph,
    DiscreteBayesianNetwork,
    GaussianBayesianNetwork,
    NaiveAdjustmentRegressor,
    NaiveIVRegressor,
)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def show_posterior(draws, estimate, title, truth=None):
    draws = np.asarray(draws, dtype=float).ravel()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(draws, bins=30, color="#4C72B0", alpha=0.9, density=True)
    ax.axvline(estimate, color="#8B1E3F", lw=2, label=f"posterior mean of the fit {estimate:.3f}")
    low, high = np.quantile(draws, [0.025, 0.975])
    ax.axvspan(low, high, color="#4C72B0", alpha=0.18, label="95% credible interval")
    if truth is not None:
        ax.axvline(truth, color="black", ls="--", lw=1.5, label=f"known effect {truth:.3f}")
    ax.set_xlabel("Effect")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
    return pd.Series({"estimate": estimate, "ci_lower": low, "ci_upper": high})



## Getting started: identification on the canonical graphs

The first checks have no data. They confirm that `CausalGraph` blocks and opens the paths the criteria use.


In [ ]:

chain = CausalGraph([("A", "B"), ("B", "C")])
fork = CausalGraph([("B", "A"), ("B", "C")])
collider = CausalGraph([("A", "B"), ("C", "B")])

rows = [
    ("chain", chain.d_separated("A", "C"), chain.d_separated("A", "C", "B")),
    ("fork", fork.d_separated("A", "C"), fork.d_separated("A", "C", "B")),
    ("collider", collider.d_separated("A", "C"), collider.d_separated("A", "C", "B")),
]
print(pd.DataFrame(rows, columns=["graph", "separated_without_B", "separated_given_B"]).to_string(index=False))

confounded = CausalGraph([("C", "T"), ("C", "Y"), ("T", "Y")])
print("Backdoor sets for T -> Y:", confounded.adjustment_sets("T", "Y"))
print("Smallest set:", confounded.minimal_adjustment_set("T", "Y"))
print("Empty set is a backdoor set:", confounded.satisfies_backdoor("T", "Y", []))

frontdoor = CausalGraph([("U", "T"), ("U", "Y"), ("T", "M"), ("M", "Y")])
print("Frontdoor sets:", frontdoor.frontdoor_sets("T", "Y"))

instrument = CausalGraph([("Z", "T"), ("U", "T"), ("U", "Y"), ("T", "Y")])
leaked = CausalGraph([("Z", "T"), ("T", "Y"), ("Z", "Y")])
print("Instruments when Z is excluded from Y:", instrument.instruments("T", "Y"))
print("Instruments when Z also points at Y:", leaked.instruments("T", "Y"))



The chain and the fork are open until $B$ is conditioned on. The collider is closed until $B$ is conditioned on. In the confounded graph the smallest backdoor set is $\{C\}$. The empty set fails because $C$ leaves $T \leftarrow C \rightarrow Y$ open. In the frontdoor graph $\{M\}$ is the mediator set, and $U$ is not an instrument. Adding $Z \rightarrow Y$ removes $Z$ from the instrument list.



## Getting started: a linear-Gaussian network with a known effect

The structural assignments are

$$
C \sim N(0,1), \qquad T = 0.5\, C + \varepsilon_T, \qquad Y = 1.5\, T - 0.7\, C + \varepsilon_Y,
$$

with $\varepsilon_T, \varepsilon_Y \sim N(0, 0.4^2)$. The interventional effect of a unit change in $T$ is $1.5$. The observational regression of $Y$ on $T$ also picks up the backdoor through $C$, so its slope is not $1.5$.


In [ ]:

rng = np.random.default_rng(0)
n = 3000
confounder = rng.normal(size=n)
treatment = 0.5 * confounder + rng.normal(scale=0.4, size=n)
outcome = 1.5 * treatment - 0.7 * confounder + rng.normal(scale=0.4, size=n)
frame = pd.DataFrame({"C": confounder, "T": treatment, "Y": outcome})

naive = np.linalg.lstsq(
    np.column_stack([np.ones(n), treatment]), outcome, rcond=None
)[0][1]

gauss = GaussianBayesianNetwork(
    [("C", "T"), ("C", "Y"), ("T", "Y")], n_draws=300, random_state=0
)
gauss.fit(frame)
ate = gauss.ate("T", "Y", treated=1.0, control=0.0)
print(f"Naive slope of Y on T: {naive:.3f}")
print("Gaussian network ATE:")
print(pd.Series(ate).round(3))
show_posterior(gauss.draws_, ate["estimate"], "Linear-Gaussian network", truth=1.5)

do1 = gauss.sample(4000, do={"T": 1.0}, random_state=1)
do0 = gauss.sample(4000, do={"T": 0.0}, random_state=2)
print(f"Sample contrast E[Y|do(T=1)] - E[Y|do(T=0)]: {(do1['Y'].mean() - do0['Y'].mean()):.3f}")



The posterior interval covers $1.5$. The naive slope does not. `sample` clamps $T$ and redraws $C$ and $Y$ from the fitted equations, so the Monte Carlo contrast estimates the same linear effect.



## Getting started: intervention versus conditioning

Binary variables with a fork at $C$:

$$
P(T=1 \mid C=1) = 0.9, \quad P(T=1 \mid C=0) = 0.1,
$$

$$
P(Y=1 \mid T, C) = 0.1 + 0.2\, T + 0.6\, C.
$$

The interventional contrast $E[Y \mid do(T=1)] - E[Y \mid do(T=0)]$ equals $0.2$, because $do(T)$ deletes the arrow $C \rightarrow T$ and the remaining coefficient on $T$ is $0.2$. Conditioning on $T=1$ leaves $C$ associated with $T$, so $P(Y=1 \mid T=1)$ is larger than $P(Y=1 \mid do(T=1))$.


In [ ]:

rng = np.random.default_rng(0)
n = 8000
confounder = rng.binomial(1, 0.5, size=n)
treatment = rng.binomial(1, np.where(confounder == 1, 0.9, 0.1))
probability = 0.1 + 0.2 * treatment + 0.6 * confounder
outcome = rng.binomial(1, probability)
binary = pd.DataFrame({"C": confounder, "T": treatment, "Y": outcome})

discrete = DiscreteBayesianNetwork(
    [("C", "T"), ("C", "Y"), ("T", "Y")], alpha=0.5, random_state=0
)
discrete.fit(binary)
print("ATE from do-queries:")
print(pd.Series(discrete.ate("T", "Y", treated=1, control=0)).round(3))

interventional = discrete.query("Y", do={"T": 1})["distributions"]["Y"][1]
conditional = discrete.query("Y", evidence={"T": 1})["distributions"]["Y"][1]
print(f"P(Y=1 | do(T=1)) = {interventional:.3f}")
print(f"P(Y=1 | T=1)     = {conditional:.3f}")
print("Query method:", discrete.query("Y", do={"T": 1})["method"])
print(discrete.cpd("Y").round(3))



## Getting started: the frontdoor functional

Simulate the graph $U \rightarrow T$, $U \rightarrow Y$, $T \rightarrow M \rightarrow Y$, with no direct arrow $T \rightarrow Y$. The `do` query on a network that still contains $U$ is the target. The frontdoor formula is then computed from the counts of $(T, M, Y)$ only.


In [ ]:

rng = np.random.default_rng(1)
n = 8000
latent = rng.binomial(1, 0.5, size=n)
treatment = rng.binomial(1, np.where(latent == 1, 0.8, 0.2))
mediator = rng.binomial(1, np.where(treatment == 1, 0.85, 0.15))
y_prob = np.where(mediator == 1, 0.15 + 0.55 * latent, 0.05 + 0.15 * latent)
outcome = rng.binomial(1, y_prob)
front = pd.DataFrame({"U": latent, "T": treatment, "M": mediator, "Y": outcome})

front_graph = CausalGraph([("U", "T"), ("U", "Y"), ("T", "M"), ("M", "Y")])
print("Frontdoor sets:", front_graph.frontdoor_sets("T", "Y"))

full = DiscreteBayesianNetwork(
    [("U", "T"), ("U", "Y"), ("T", "M"), ("M", "Y")], alpha=0.5, random_state=0
)
full.fit(front)
target = full.ate("T", "Y", treated=1, control=0)["estimate"]


def frontdoor_ate(data: pd.DataFrame) -> float:
    # Pearl's frontdoor contrast for binary T, M, Y. The latent U is not used.
    t = data["T"].to_numpy()
    m = data["M"].to_numpy()
    y = data["Y"].to_numpy()

    def pt(value):
        return float(np.mean(t == value))

    def pm_given_t(m_value, t_value):
        selected = t == t_value
        return float(np.mean(m[selected] == m_value))

    def py_given_mt(y_value, m_value, t_value):
        selected = (m == m_value) & (t == t_value)
        return float(np.mean(y[selected] == y_value))

    def mean_y(t_value):
        total = 0.0
        for m_value in (0, 1):
            inner = 0.0
            for t_prime in (0, 1):
                inner += py_given_mt(1, m_value, t_prime) * pt(t_prime)
            total += pm_given_t(m_value, t_value) * inner
        return total

    return mean_y(1) - mean_y(0)


print(f"ATE from the network that still contains U: {target:.3f}")
print(f"ATE from the frontdoor formula on (T, M, Y): {frontdoor_ate(front):.3f}")



## Getting started: adjustment and instrumental variables

Two linear systems with effect $1.5$. The first is the backdoor graph $C \rightarrow T$, $C \rightarrow Y$, $T \rightarrow Y$. The second is $Z \rightarrow T \rightarrow Y$ with a latent common cause of $T$ and $Y$ that is present in the data-generating process and absent from $Z$'s exclusion path.


In [ ]:

rng = np.random.default_rng(1)
n = 2500
confounder = rng.normal(size=n)
treatment = 0.8 * confounder + rng.normal(scale=0.5, size=n)
outcome = 1.5 * treatment + 1.2 * confounder + rng.normal(scale=0.4, size=n)
adjust_frame = pd.DataFrame({"C": confounder, "T": treatment, "Y": outcome})
adjusted = NaiveAdjustmentRegressor(
    CausalGraph([("C", "T"), ("C", "Y"), ("T", "Y")]),
    "T",
    "Y",
    n_draws=400,
    random_state=0,
)
adjusted.fit(adjust_frame)
print("Adjustment set:", adjusted.adjustment_set_)
print(pd.Series(adjusted.summary_).round(3))
show_posterior(adjusted.draws_, adjusted.effect_, "Naive adjustment", truth=1.5)

instrument = rng.normal(size=n)
latent = rng.normal(size=n)
endogenous = 0.9 * instrument + 0.7 * latent + rng.normal(scale=0.3, size=n)
response = 1.5 * endogenous + 0.8 * latent + rng.normal(scale=0.3, size=n)
iv_frame = pd.DataFrame({"Z": instrument, "T": endogenous, "Y": response})
iv_graph = CausalGraph([("Z", "T"), ("T", "Y")])
print("Instruments on the drawn graph:", iv_graph.instruments("T", "Y"))
naive_iv = NaiveIVRegressor(iv_graph, "T", "Y", n_draws=80, random_state=0)
naive_iv.fit(iv_frame)
print("Instruments:", naive_iv.instruments_)
print(pd.Series(naive_iv.summary_).round(3))
show_posterior(naive_iv.draws_, naive_iv.effect_, "Naive IV", truth=1.5)



### Key parameters

| Parameter | Class | Role |
|---|---|---|
| `edges` | graph, both networks | `(parent, child)` pairs. A cycle is rejected. |
| `prior_precision` | Gaussian network, adjustment | Ridge penalty on slopes. The default is a weak prior. |
| `n_draws` | Gaussian network, adjustment, IV | Number of posterior draws for the interval. |
| `alpha` | discrete network | Dirichlet concentration added to every cell. |
| `exact_limit` | discrete network | Largest joint table that is summed exactly. |
| `random_state` | all stochastic fits | Seed for posterior draws and likelihood weighting. |



## Use case: the Sachs protein-signaling network

The measurements are the flow-cytometry sample shipped as `Data/sachs_data.csv`: 11 phosphoproteins and phospholipids, one row per cell. Sachs et al. (2005) introduced this system as a Bayesian-network benchmark. This notebook does not rediscover the edges. It conditions on a standard published skeleton — the same directed edges used with the Sachs sample in the bnlearn documentation — and asks two questions the classes are built to answer.

1. Which observed proteins form a backdoor set for the effect of Mek on Erk?
2. What is the posterior of that total effect on the log scale, and how does the single-protein regression compare?

Proteins are analyzed as $\log x$. The Gaussian effect is then a log-log slope: the percent change in Erk associated with a one-percent interventional change in Mek, along the directed paths in this skeleton. Erk's other parent in the skeleton is PKA, and every backdoor from Mek into Erk passes through PKA, so the smallest adjustment set is `{pka}`.



### Data

Eleven columns: `raf`, `mek`, `plc`, `pip2`, `pip3`, `erk`, `akt`, `pka`, `pkc`, `p38`, `jnk`. Values are continuous abundances. There is no treatment indicator in the file. The intervention is the `do` operator on the fitted structural equations.


In [ ]:

sachs = pd.read_csv(data_dir / "sachs_data.csv")
print("Shape:", sachs.shape)
print(sachs.describe().T[["mean", "std", "min", "max"]].round(2))

proteins = ["plc", "pip2", "pip3", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]
sachs_log = np.log(sachs[proteins].clip(lower=1e-6))
sachs_log.describe().T[["mean", "std"]].round(3)


### Data visualization


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(sachs_log["mek"], sachs_log["erk"], s=6, alpha=0.25, c="#4C72B0")
axes[0].set_xlabel("log Mek")
axes[0].set_ylabel("log Erk")
axes[0].set_title("Observational cloud")
axes[1].scatter(sachs_log["pka"], sachs_log["erk"], s=6, alpha=0.25, c="#8B1E3F")
axes[1].set_xlabel("log PKA")
axes[1].set_ylabel("log Erk")
axes[1].set_title("PKA is Erk's other parent in the skeleton")
fig.tight_layout()
plt.show()



### The skeleton and the adjustment set

Edges point from the upstream kinase or phospholipid to its published target. `CausalGraph` rejects the list if it contains a cycle.


In [ ]:

sachs_edges = [
    ("pkc", "pka"), ("pkc", "raf"), ("pkc", "mek"), ("pkc", "jnk"), ("pkc", "p38"),
    ("pka", "raf"), ("pka", "mek"), ("pka", "erk"), ("pka", "akt"), ("pka", "jnk"), ("pka", "p38"),
    ("raf", "mek"), ("mek", "erk"),
    ("plc", "pkc"), ("plc", "pip3"), ("plc", "pip2"),
    ("pip3", "pip2"), ("pip2", "pkc"), ("pip3", "akt"),
]
sachs_graph = CausalGraph(sachs_edges)
print("Nodes in topological order:", sachs_graph.nodes)
print("Parents of Mek:", sachs_graph.parents("mek"))
print("Parents of Erk:", sachs_graph.parents("erk"))
print("Minimal backdoor set, Mek -> Erk:", sachs_graph.minimal_adjustment_set("mek", "erk"))
print("All minimal backdoor sets:", sachs_graph.adjustment_sets("mek", "erk"))
print("Instruments for Mek -> Erk:", sachs_graph.instruments("mek", "erk"))
print("Frontdoor sets:", sachs_graph.frontdoor_sets("mek", "erk"))

adjacency = sachs_graph.adjacency()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
image = ax.imshow(adjacency.to_numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(adjacency.columns)))
ax.set_yticks(range(len(adjacency.index)))
ax.set_xticklabels(adjacency.columns, rotation=90)
ax.set_yticklabels(adjacency.index)
ax.set_title("Adjacency: row child, column parent")
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()



The instrument list is empty: no protein reaches Mek only through a path that is blocked from Erk once Mek is deleted. The frontdoor list is empty: Mek's only directed path to Erk is the direct edge, and that edge is not a mediator set. Identification is by adjustment. The smallest set is PKA.


### Gaussian network and backdoor regression


In [ ]:

sachs_net = GaussianBayesianNetwork(sachs_edges, n_draws=300, random_state=0)
sachs_net.fit(sachs_log)
sachs_ate = sachs_net.ate("mek", "erk")
print("Total effect of log Mek on log Erk:")
print(pd.Series(sachs_ate).round(3))

erk_equation = sachs_net.equations_["erk"]
print("Erk equation features:", erk_equation["feature_names"])
print("Erk posterior-mean coefficients:", np.round(erk_equation["coef"], 3))

naive_slope = np.linalg.lstsq(
    np.column_stack([np.ones(len(sachs_log)), sachs_log["mek"]]),
    sachs_log["erk"],
    rcond=None,
)[0][1]
print(f"Naive slope of log Erk on log Mek: {naive_slope:.3f}")

sachs_adj = NaiveAdjustmentRegressor(
    sachs_graph, "mek", "erk", n_draws=400, random_state=0
)
sachs_adj.fit(sachs_log)
print("Backdoor set:", sachs_adj.adjustment_set_)
print(pd.Series(sachs_adj.summary_).round(3))
show_posterior(
    sachs_net.draws_,
    sachs_ate["estimate"],
    "Sachs skeleton: total effect of Mek on Erk",
)



Mek's only child in this skeleton is Erk, so the total effect equals the direct coefficient on Mek in the Erk equation. The backdoor regression of Erk on Mek and PKA estimates that same coefficient. The single-protein slope is close to it in this sample: the observational association of Mek and Erk is not dominated by the PKA backdoor. Closeness is a result about these measurements. It is not a reason to skip the graph. A different skeleton, or a different pair of proteins, moves the two numbers apart, as the simulated fork did.



### A discrete slice of the same cells

Dichotomize PKA, Mek, and Erk at their medians and fit the fork `pka → mek`, `pka → erk`, `mek → erk`. The outcome is a 0/1 label, so `ate` is a difference in probabilities, not a log-abundance slope. The comparison that matters is `do(mek=1)` against conditioning on `mek=1`.


In [ ]:

discrete_sachs = pd.DataFrame({
    "pka": (sachs_log["pka"] >= sachs_log["pka"].median()).astype(int),
    "mek": (sachs_log["mek"] >= sachs_log["mek"].median()).astype(int),
    "erk": (sachs_log["erk"] >= sachs_log["erk"].median()).astype(int),
})
slice_net = DiscreteBayesianNetwork(
    [("pka", "mek"), ("pka", "erk"), ("mek", "erk")], alpha=1.0, random_state=0
)
slice_net.fit(discrete_sachs)
print(pd.Series(slice_net.ate("mek", "erk", treated=1, control=0)).round(3))
print(
    "P(Erk high | do(Mek high)) =",
    round(slice_net.query("erk", do={"mek": 1})["distributions"]["erk"][1], 3),
)
print(
    "P(Erk high | Mek high)     =",
    round(slice_net.query("erk", evidence={"mek": 1})["distributions"]["erk"][1], 3),
)



### How to read the Sachs fit

In the median-split slice, $P(\text{Erk high} \mid do(\text{Mek high}))$ and $P(\text{Erk high} \mid \text{Mek high})$ are nearly the same. The simulated fork is the case where those two probabilities separate. The dichotomized Sachs slice is a weak fork, which is a result about the split, not a reason to skip the comparison.

- The adjacency matrix is an assumption. `ate` propagates coefficient uncertainty through that assumption. It does not propagate uncertainty about which edges exist.
- The credible interval is a statement about the linear coefficients given the skeleton and the ridge prior. A nonlinear dependence among the proteins is outside the model.
- Median splits throw away the abundance scale. Use them to compare `do` with conditioning. Use the Gaussian network for the log-log effect.
- An empty instrument list is a finding about this skeleton. It says the file, under these edges, does not contain a protein that can be an instrument for Mek.



## Summary and conclusion

`CausalGraph` answers the identification question before any likelihood is fit: which set blocks the backdoors, which mediators satisfy the frontdoor criterion, and which nodes are instruments. `GaussianBayesianNetwork` turns a linear skeleton into a posterior for $E[Y \mid do(T)]$. `DiscreteBayesianNetwork` keeps the distinction between clamping a node and conditioning on it. `NaiveAdjustmentRegressor` and `NaiveIVRegressor` are the two regressions those criteria license, with a normal or Bayesian-bootstrap posterior on the coefficient.

On the simulated fork the naive slope and the interventional effect separate cleanly, and the posterior covers the known value $1.5$. On the Sachs skeleton the smallest backdoor set for Mek on Erk is PKA, the total effect equals the direct coefficient, and the interval sits on a small negative log-log slope. The graph is an input. The posterior describes the coefficients given that input.



## Scientific terminology

| Term | Meaning in this notebook |
|---|---|
| DAG | A directed graph with no directed cycles. |
| d-separation | A graphical criterion for conditional independence in a DAG. |
| Backdoor path | A path from treatment to outcome that starts with an arrow into the treatment. |
| Adjustment set | A covariate set that blocks every backdoor and contains no descendant of treatment. |
| Frontdoor | Identification through a mediator when treatment and outcome share an unobserved common cause. |
| $do(T=t)$ | A hard intervention that replaces the structural equation for $T$ with the constant $t$. |
| Total effect | The sum of the products of coefficients along every directed path from $T$ to $Y$. |
| Credible interval | The central 95% of posterior draws of the effect. |
| Dirichlet prior | A prior on a probability vector. The posterior mean shrinks the empirical frequencies toward a uniform table. |
